# RQ3-01 — Prepare 1,000 Human Holdout & Leakage-Free Retraining Data

Notebook này tạo dữ liệu nền cho RQ3 theo đúng thiết kế:

- Chọn **1,000 Human-written essays** từ PERSUADE: **10 prompts × 100 bài/prompt**.
- Chỉ chọn Human essay có mặt trong `data_with_features.csv`, để việc loại khỏi dataset train là xác định.
- Xuất `rq3_human_holdout_1000.csv` — **chỉ dùng RQ3 external test**.
- Xuất `processed_data_minus_1000_human.csv`.
- Xuất `data_with_features_minus_1000_human.csv` — file dùng để **retrain toàn bộ detector**.
- Audit bằng SHA-256 hash để đảm bảo 1,000 Human holdout không còn trong training dataset.

> Quy tắc: `rq3_human_holdout_1000.csv` tuyệt đối không được dùng để train, chọn epoch, tune threshold hay tune hyperparameter.

In [ ]:
import csv
import hashlib
import random
import re
from pathlib import Path

SEED = 42
PER_PROMPT = 100

cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name.lower() == "notebooks" else cwd
if Path("/content/AI-Generated-Text-Essay-Detection").exists():
    PROJECT_ROOT = Path("/content/AI-Generated-Text-Essay-Detection")

PROCESSED_CANDIDATES = [
    PROJECT_ROOT / "data" / "processed" / "processed_data.csv",
    PROJECT_ROOT / "processed_data.csv",
    cwd / "processed_data.csv",
]
FEATURE_CANDIDATES = [
    PROJECT_ROOT / "data" / "processed" / "data_with_features.csv",
    PROJECT_ROOT / "data_with_features.csv",
    cwd / "data_with_features.csv",
]

PROCESSED_FILE = next((p for p in PROCESSED_CANDIDATES if p.exists()), None)
FEATURE_FILE = next((p for p in FEATURE_CANDIDATES if p.exists()), None)
if PROCESSED_FILE is None or FEATURE_FILE is None:
    raise FileNotFoundError(
        f"Không tìm thấy input files.\nprocessed={PROCESSED_FILE}\nfeatures={FEATURE_FILE}"
    )

RQ3_DIR = PROJECT_ROOT / "data" / "rq3"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RQ3_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

HOLDOUT_OUT = RQ3_DIR / "rq3_human_holdout_1000.csv"
PROCESSED_OUT = PROCESSED_DIR / "processed_data_minus_1000_human.csv"
FEATURES_OUT = PROCESSED_DIR / "data_with_features_minus_1000_human.csv"

SELECTED_PROMPTS = [
    "Phones and driving",
    "Distance learning",
    "Summer projects",
    "Mandatory extracurricular activities",
    "Cell phones at school",
    "Grades for extracurricular activities",
    "Seeking multiple opinions",
    "Community service",
    "Driverless cars",
    "Does the electoral college work?",
]

print("PROCESSED_FILE:", PROCESSED_FILE)
print("FEATURE_FILE  :", FEATURE_FILE)
print("Outputs       :", RQ3_DIR, PROCESSED_DIR)

In [ ]:
def normalize_text(text):
    return re.sub(r"\s+", " ", str(text).strip().lower())

def text_hash(text):
    return hashlib.sha256(normalize_text(text).encode("utf-8")).hexdigest()

# Inventory toàn bộ text trong feature dataset.
feature_hash_to_count = {}
feature_total = 0
feature_human_before = 0
feature_ai_before = 0

with FEATURE_FILE.open("r", encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f)
    required = {"text", "label"}
    if not required.issubset(reader.fieldnames or []):
        raise ValueError(f"Feature file thiếu cột: {required - set(reader.fieldnames or [])}")
    for row in reader:
        feature_total += 1
        label = str(row["label"]).strip()
        feature_human_before += int(label == "0")
        feature_ai_before += int(label == "1")
        h = text_hash(row["text"])
        feature_hash_to_count[h] = feature_hash_to_count.get(h, 0) + 1

print("Feature rows:", feature_total)
print("Human:", feature_human_before, "AI:", feature_ai_before)

In [ ]:
# Thu candidate Human từ PERSUADE, bắt buộc có đúng 1 match trong feature dataset.
candidates = {p: [] for p in SELECTED_PROMPTS}

with PROCESSED_FILE.open("r", encoding="utf-8", newline="") as f:
    reader = csv.DictReader(f)
    required = {"label", "source", "prompt_name", "text_clean", "word_count_clean"}
    if not required.issubset(reader.fieldnames or []):
        raise ValueError(f"processed_data.csv thiếu cột: {required - set(reader.fieldnames or [])}")

    for row in reader:
        prompt = row["prompt_name"]
        if (
            row["label"] == "0"
            and row["source"] == "persuade_corpus"
            and prompt in candidates
        ):
            h = text_hash(row["text_clean"])
            if feature_hash_to_count.get(h, 0) == 1:
                x = dict(row)
                x["text_hash"] = h
                candidates[prompt].append(x)

for prompt in SELECTED_PROMPTS:
    unique_n = len({r["text_hash"] for r in candidates[prompt]})
    print(f"{prompt:45s}: {unique_n:4d} eligible Human")
    if unique_n < PER_PROMPT:
        raise RuntimeError(f"Không đủ {PER_PROMPT} Human cho prompt: {prompt}")

In [ ]:
rng = random.Random(SEED)
selected_rows = []
selected_hashes = set()

for prompt in SELECTED_PROMPTS:
    unique = {}
    for row in candidates[prompt]:
        unique.setdefault(row["text_hash"], row)
    chosen = rng.sample(list(unique.values()), PER_PROMPT)
    for row in chosen:
        if row["text_hash"] in selected_hashes:
            raise RuntimeError("Duplicate holdout hash across prompts")
        selected_hashes.add(row["text_hash"])
    selected_rows.extend(chosen)

assert len(selected_rows) == 1000
assert len(selected_hashes) == 1000

holdout_fields = [
    "essay_id", "label", "prompt_name", "source",
    "text_clean", "word_count_clean", "text_hash"
]
with HOLDOUT_OUT.open("w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=holdout_fields)
    writer.writeheader()
    for i, row in enumerate(selected_rows, 1):
        writer.writerow({
            "essay_id": f"H_RQ3_{i:04d}",
            "label": 0,
            "prompt_name": row["prompt_name"],
            "source": row["source"],
            "text_clean": row["text_clean"],
            "word_count_clean": row["word_count_clean"],
            "text_hash": row["text_hash"],
        })
print("Saved:", HOLDOUT_OUT)

In [ ]:
# Loại holdout khỏi processed_data.csv
processed_before = processed_after = processed_removed = 0
processed_ai_before = processed_ai_after = 0
processed_human_before = processed_human_after = 0

with PROCESSED_FILE.open("r", encoding="utf-8", newline="") as fin, \
     PROCESSED_OUT.open("w", encoding="utf-8", newline="") as fout:
    reader = csv.DictReader(fin)
    writer = csv.DictWriter(fout, fieldnames=reader.fieldnames)
    writer.writeheader()

    for row in reader:
        processed_before += 1
        label = str(row["label"]).strip()
        processed_human_before += int(label == "0")
        processed_ai_before += int(label == "1")
        h = text_hash(row["text_clean"])
        if h in selected_hashes:
            if label != "0":
                raise RuntimeError("Holdout hash matched an AI row")
            processed_removed += 1
            continue
        writer.writerow(row)
        processed_after += 1
        processed_human_after += int(label == "0")
        processed_ai_after += int(label == "1")

# Loại cùng 1,000 hash khỏi data_with_features.csv
feature_after = feature_removed = 0
feature_human_after = feature_ai_after = 0
with FEATURE_FILE.open("r", encoding="utf-8-sig", newline="") as fin, \
     FEATURES_OUT.open("w", encoding="utf-8-sig", newline="") as fout:
    reader = csv.DictReader(fin)
    writer = csv.DictWriter(fout, fieldnames=reader.fieldnames)
    writer.writeheader()

    for row in reader:
        label = str(row["label"]).strip()
        h = text_hash(row["text"])
        if h in selected_hashes:
            if label != "0":
                raise RuntimeError("Holdout hash matched an AI feature row")
            feature_removed += 1
            continue
        writer.writerow(row)
        feature_after += 1
        feature_human_after += int(label == "0")
        feature_ai_after += int(label == "1")

assert processed_removed == 1000
assert feature_removed == 1000
assert processed_ai_before == processed_ai_after
assert feature_ai_before == feature_ai_after

print(f"Processed: {processed_before:,} -> {processed_after:,}")
print(f"Features : {feature_total:,} -> {feature_after:,}")
print(f"Feature labels: Human {feature_human_before:,}->{feature_human_after:,}; AI {feature_ai_before:,}->{feature_ai_after:,}")
print("Saved:", PROCESSED_OUT)
print("Saved:", FEATURES_OUT)

In [ ]:
# Final leakage audit: không có holdout hash trong training feature file.
remaining_hashes = set()
with FEATURES_OUT.open("r", encoding="utf-8-sig", newline="") as f:
    for row in csv.DictReader(f):
        remaining_hashes.add(text_hash(row["text"]))

overlap = selected_hashes & remaining_hashes
assert len(overlap) == 0

prompt_counts = {}
for row in selected_rows:
    prompt_counts[row["prompt_name"]] = prompt_counts.get(row["prompt_name"], 0) + 1
assert all(prompt_counts[p] == 100 for p in SELECTED_PROMPTS)

print("✅ LEAKAGE AUDIT PASS")
print("Holdout rows:", len(selected_rows))
print("Overlap with retraining features:", len(overlap))
for p in SELECTED_PROMPTS:
    print(f"{p:45s}: {prompt_counts[p]}")